In [1]:


import duckdb
import pandas as pd
from datetime import datetime

con = duckdb.connect("ecommerce.duckdb")

print("Conexión realizada")


Conexión realizada


In [2]:
ruta = "./data/"

#Carga de datos RAW

con.execute(f"""
CREATE OR REPLACE TABLE orders_raw AS
SELECT * FROM read_csv_auto('{ruta}olist_orders_dataset.csv', all_varchar=true);
""")

con.execute(f"""
CREATE OR REPLACE TABLE order_items_raw AS
SELECT * FROM read_csv_auto('{ruta}olist_order_items_dataset.csv', all_varchar=true);
""")

con.execute(f"""
CREATE OR REPLACE TABLE customers_raw AS
SELECT * FROM read_csv_auto('{ruta}olist_customers_dataset.csv', all_varchar=true);
""")

con.execute(f"""
CREATE OR REPLACE TABLE payments_raw AS
SELECT * FROM read_csv_auto('{ruta}olist_order_payments_dataset.csv', all_varchar=true);
""")

con.execute(f"""
CREATE OR REPLACE TABLE products_raw AS
SELECT * FROM read_csv_auto('{ruta}olist_products_dataset.csv', all_varchar=true);
""")

In [3]:
# Ver cantidad de registros de cada tabla
print(con.execute("SELECT 'orders' AS tabla, COUNT(*) AS registros FROM orders_raw").df())
print(con.execute("SELECT 'order_items' AS tabla, COUNT(*) AS registros FROM order_items_raw").df())
print(con.execute("SELECT 'customers' AS tabla, COUNT(*) AS registros FROM customers_raw").df())
print(con.execute("SELECT 'payments' AS tabla, COUNT(*) AS registros FROM payments_raw").df())
print(con.execute("SELECT 'products' AS tabla, COUNT(*) AS registros FROM products_raw").df())

    tabla  registros
0  orders      99441
         tabla  registros
0  order_items     112650
       tabla  registros
0  customers      99441
      tabla  registros
0  payments     103886
      tabla  registros
0  products      32951


In [4]:

# cantidad de registros

print("\n Cantidad de registros por tabla")
display(con.execute("""
SELECT 'orders_raw' AS tabla, COUNT(*) AS registros FROM orders_raw
UNION ALL
SELECT 'order_items_raw', COUNT(*) FROM order_items_raw
UNION ALL
SELECT 'customers_raw', COUNT(*) FROM customers_raw
UNION ALL
SELECT 'payments_raw', COUNT(*) FROM payments_raw
UNION ALL
SELECT 'products_raw', COUNT(*) FROM products_raw
""").df())


print("\n Nulos y vacios")

print("\n ORDERS_RAW ")
display(con.execute("""
SELECT 
    COUNT(*) FILTER (WHERE order_id IS NULL OR TRIM(order_id) = '') AS order_id,
    COUNT(*) FILTER (WHERE customer_id IS NULL OR TRIM(customer_id) = '') AS customer_id,
    COUNT(*) FILTER (WHERE order_status IS NULL OR TRIM(order_status) = '') AS order_status,
    COUNT(*) FILTER (WHERE order_purchase_timestamp IS NULL OR TRIM(order_purchase_timestamp) = '') AS purchase_ts,
    COUNT(*) FILTER (WHERE order_approved_at IS NULL OR TRIM(order_approved_at) = '') AS approved_at,
    COUNT(*) FILTER (WHERE order_delivered_carrier_date IS NULL OR TRIM(order_delivered_carrier_date) = '') AS carrier_date,
    COUNT(*) FILTER (WHERE order_delivered_customer_date IS NULL OR TRIM(order_delivered_customer_date) = '') AS delivered_date,
    COUNT(*) FILTER (WHERE order_estimated_delivery_date IS NULL OR TRIM(order_estimated_delivery_date) = '') AS estimated_date
FROM orders_raw
""").df())

print("\n ORDER_ITEMS_RAW")
display(con.execute("""
SELECT 
    COUNT(*) FILTER (WHERE order_id IS NULL OR TRIM(order_id) = '') AS order_id,
    COUNT(*) FILTER (WHERE order_item_id IS NULL OR TRIM(order_item_id) = '') AS order_item_id,
    COUNT(*) FILTER (WHERE product_id IS NULL OR TRIM(product_id) = '') AS product_id,
    COUNT(*) FILTER (WHERE seller_id IS NULL OR TRIM(seller_id) = '') AS seller_id,
    COUNT(*) FILTER (WHERE price IS NULL OR TRIM(price) = '') AS price,
    COUNT(*) FILTER (WHERE freight_value IS NULL OR TRIM(freight_value) = '') AS freight_value
FROM order_items_raw
""").df())

print("\n CUSTOMERS_RAW")
display(con.execute("""
SELECT 
    COUNT(*) FILTER (WHERE customer_id IS NULL OR TRIM(customer_id) = '') AS customer_id,
    COUNT(*) FILTER (WHERE customer_unique_id IS NULL OR TRIM(customer_unique_id) = '') AS customer_unique_id,
    COUNT(*) FILTER (WHERE customer_zip_code_prefix IS NULL OR TRIM(customer_zip_code_prefix) = '') AS zip_code,
    COUNT(*) FILTER (WHERE customer_city IS NULL OR TRIM(customer_city) = '') AS city,
    COUNT(*) FILTER (WHERE customer_state IS NULL OR TRIM(customer_state) = '') AS state
FROM customers_raw
""").df())

print("\n PAYMENTS_RAW")
display(con.execute("""
SELECT 
    COUNT(*) FILTER (WHERE order_id IS NULL OR TRIM(order_id) = '') AS order_id,
    COUNT(*) FILTER (WHERE payment_type IS NULL OR TRIM(payment_type) = '') AS payment_type,
    COUNT(*) FILTER (WHERE payment_value IS NULL OR TRIM(payment_value) = '') AS payment_value
FROM payments_raw
""").df())

print("\n PRODUCTS_RAW ")
display(con.execute("""
SELECT 
    COUNT(*) FILTER (WHERE product_id IS NULL OR TRIM(product_id) = '') AS product_id,
    COUNT(*) FILTER (WHERE product_category_name IS NULL OR TRIM(product_category_name) = '') AS category_name
FROM products_raw
""").df())

print("\n Duplicados")

print("\nDuplicados de order_id en orders_raw:")
display(con.execute("""
SELECT COUNT(*) AS pedidos_duplicados
FROM (
    SELECT order_id 
    FROM orders_raw 
    GROUP BY order_id 
    HAVING COUNT(*) > 1
)
""").df())

print("\nDuplicados de (order_id + order_item_id) en order_items_raw:")
display(con.execute("""
SELECT COUNT(*) AS items_duplicados
FROM (
    SELECT order_id, order_item_id
    FROM order_items_raw 
    GROUP BY order_id, order_item_id 
    HAVING COUNT(*) > 1
)
""").df())


print("\n Estado de pedido")
display(con.execute("""
SELECT order_status, COUNT(*) AS cantidad,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM orders_raw
GROUP BY order_status
ORDER BY cantidad DESC
""").df())

print("\n Pagos")
display(con.execute("""
SELECT payment_type, COUNT(*) AS cantidad
FROM payments_raw
GROUP BY payment_type
ORDER BY cantidad DESC
""").df())

print("\n Top categorias de producto")
display(con.execute("""
SELECT product_category_name, COUNT(*) AS cantidad
FROM products_raw
GROUP BY product_category_name
ORDER BY cantidad DESC
LIMIT 10
""").df())


 Cantidad de registros por tabla


,tabla,registros
0,orders_raw,99441
1,order_items_raw,112650
2,customers_raw,99441
3,payments_raw,103886
4,products_raw,32951



 Nulos y vacios

 ORDERS_RAW 


,order_id,customer_id,order_status,purchase_ts,approved_at,carrier_date,delivered_date,estimated_date
0,0,0,0,0,160,1783,2965,0



 ORDER_ITEMS_RAW


,order_id,order_item_id,product_id,seller_id,price,freight_value
0,0,0,0,0,0,0


... (12 outputs left out)


 Pagos


,payment_type,cantidad
0,credit_card,76795
1,boleto,19784
2,voucher,5775
3,debit_card,1529
4,not_defined,3



 Top categorias de producto


,product_category_name,cantidad
0,cama_mesa_banho,3029
1,esporte_lazer,2867
2,moveis_decoracao,2657
3,beleza_saude,2444
4,utilidades_domesticas,2335
5,automotivo,1900
6,informatica_acessorios,1639
7,brinquedos,1411
8,relogios_presentes,1329
9,telefonia,1134


In [5]:
## Creación de errores controlados en nuevas tablas


In [6]:
# copia de tablas originales

con.execute("CREATE OR REPLACE TABLE orders_dirty AS SELECT * FROM orders_raw")
con.execute("CREATE OR REPLACE TABLE order_items_dirty AS SELECT * FROM order_items_raw")
con.execute("CREATE OR REPLACE TABLE customers_dirty AS SELECT * FROM customers_raw")
con.execute("CREATE OR REPLACE TABLE payments_dirty AS SELECT * FROM payments_raw")
con.execute("CREATE OR REPLACE TABLE products_dirty AS SELECT * FROM products_raw")

In [7]:
# ERRORES CONTROLADOS

In [8]:
# orders

# Modificar 70 fechas de entrega
con.execute("""
    UPDATE orders_dirty
    SET order_delivered_customer_date = '2016-01-15 10:00:00'
    WHERE order_id IN (
        SELECT order_id
        FROM orders_raw
        ORDER BY order_id
        LIMIT 70
    )
""")

# Cambiar 35 pedidos entregados a canceled.
con.execute("""
    UPDATE orders_dirty
    SET order_status = 'canceled'
    WHERE order_id IN (
        SELECT order_id
        FROM orders_raw
        WHERE order_status = 'delivered'
        ORDER BY order_id
        LIMIT 35 OFFSET 70
    )
""")

# Agregar espacios a 25 estados.
# Se excluyen los primeros 105 pedidos entregados.
con.execute("""
    UPDATE orders_dirty
    SET order_status = '  delivered  '
    WHERE order_id IN (
        SELECT order_id
        FROM orders_raw
        WHERE order_status = 'delivered'
        ORDER BY order_id
        LIMIT 25 OFFSET 105
    )
""")

# Insertar 50 pedidos duplicados
con.execute("""
    INSERT INTO orders_dirty
    SELECT *
    FROM orders_raw
    ORDER BY order_id
    LIMIT 50
""")

print("Errores controlados de orders aplicados.")

Errores controlados de orders aplicados.


In [9]:
# order_items

#  precios negativos
con.execute("""
    UPDATE order_items_dirty
    SET price = '-25.90'
    WHERE (order_id, order_item_id) IN (
        SELECT order_id, order_item_id
        FROM order_items_raw
        ORDER BY order_id, order_item_id
        LIMIT 40
    )
""")

# precios iguales a cero
con.execute("""
    UPDATE order_items_dirty
    SET price = '0'
    WHERE (order_id, order_item_id) IN (
        SELECT order_id, order_item_id
        FROM order_items_raw
        ORDER BY order_id, order_item_id
        LIMIT 20 OFFSET 40
    )
""")

# valores de flete negativos
con.execute("""
    UPDATE order_items_dirty
    SET freight_value = '-12.50'
    WHERE (order_id, order_item_id) IN (
        SELECT order_id, order_item_id
        FROM order_items_raw
        ORDER BY order_id, order_item_id
        LIMIT 30 OFFSET 60
    )
""")

# registros duplicados
con.execute("""
    INSERT INTO order_items_dirty
    SELECT *
    FROM order_items_raw
    ORDER BY order_id, order_item_id
    LIMIT 35
""")


In [10]:
# customers

# ciudades con espacios y mayúsculas
con.execute("""
    UPDATE customers_dirty
    SET customer_city = '  SAO PAULO  '
    WHERE customer_id IN (
        SELECT customer_id
        FROM customers_raw
        ORDER BY customer_id
        LIMIT 50
    )
""")

# códigos postales demasiado cortos
con.execute("""
    UPDATE customers_dirty
    SET customer_zip_code_prefix = '123'
    WHERE customer_id IN (
        SELECT customer_id
        FROM customers_raw
        ORDER BY customer_id
        LIMIT 25
    )
""")

# códigos postales alfanuméricos
con.execute("""
    UPDATE customers_dirty
    SET customer_zip_code_prefix = 'ABC12'
    WHERE customer_id IN (
        SELECT customer_id
        FROM customers_raw
        ORDER BY customer_id
        LIMIT 15 OFFSET 25
    )
""")



In [11]:
# payments

# espacios adicionales en 40 tipos de pago
con.execute("""
    UPDATE payments_dirty
    SET payment_type = 'credit_card '
    WHERE (order_id, payment_sequential) IN (
        SELECT order_id, payment_sequential
        FROM payments_raw
        ORDER BY order_id, payment_sequential
        LIMIT 40
    )
""")

# denominaciones incorrectas
con.execute("""
    UPDATE payments_dirty
    SET payment_type = 'boleto_bancario'
    WHERE (order_id, payment_sequential) IN (
        SELECT order_id, payment_sequential
        FROM payments_raw
        ORDER BY order_id, payment_sequential
        LIMIT 25 OFFSET 40
    )
""")

# tipos de pago no válidos
con.execute("""
    UPDATE payments_dirty
    SET payment_type = 'otro'
    WHERE (order_id, payment_sequential) IN (
        SELECT order_id, payment_sequential
        FROM payments_raw
        ORDER BY order_id, payment_sequential
        LIMIT 20 OFFSET 65
    )
""")


In [12]:
# products

#  espacios adicionales
con.execute("""
    UPDATE products_dirty
    SET product_category_name = '  cama_mesa_banho  '
    WHERE product_id IN (
        SELECT product_id
        FROM products_raw
        ORDER BY product_id
        LIMIT 35
    )
""")

#  categorías en mayúsculas
con.execute("""
    UPDATE products_dirty
    SET product_category_name = 'CAMA_MESA_BANHO'
    WHERE product_id IN (
        SELECT product_id
        FROM products_raw
        ORDER BY product_id
        LIMIT 25 OFFSET 35
    )
""")



In [13]:
# VERIFICACIÓN DE ERRORES


In [14]:
# Pedidos
print("PEDIDOS")
print("Duplicados:")
display(con.execute("""
    SELECT COUNT(*) AS grupos_duplicados
    FROM (
        SELECT order_id
        FROM orders_dirty
        GROUP BY order_id
        HAVING COUNT(*) > 1
    )
""").df())

print("Estados con espacios:")
display(con.execute("""
    SELECT COUNT(*) AS registros
    FROM orders_dirty
    WHERE order_status = '  delivered  '
""").df())

# Artículos
print("ARTÍCULOS")
display(con.execute("""
    SELECT
        SUM(CASE WHEN TRY_CAST(price AS DOUBLE) < 0 THEN 1 ELSE 0 END)
            AS precios_negativos,
        SUM(CASE WHEN TRY_CAST(price AS DOUBLE) = 0 THEN 1 ELSE 0 END)
            AS precios_cero,
        SUM(CASE WHEN TRY_CAST(freight_value AS DOUBLE) < 0 THEN 1 ELSE 0 END)
            AS fletes_negativos
    FROM order_items_dirty
""").df())

# Clientes
print("CLIENTES")
display(con.execute("""
    SELECT
        SUM(CASE WHEN customer_city = '  SAO PAULO  ' THEN 1 ELSE 0 END)
            AS ciudades_con_espacios,
        SUM(CASE WHEN customer_zip_code_prefix = '123' THEN 1 ELSE 0 END)
            AS zip_cortos,
        SUM(CASE WHEN customer_zip_code_prefix = 'ABC12' THEN 1 ELSE 0 END)
            AS zip_no_numericos
    FROM customers_dirty
""").df())

# Pagos
print("PAGOS")
display(con.execute("""
    SELECT
        payment_type,
        COUNT(*) AS cantidad
    FROM payments_dirty
    WHERE payment_type IN ('credit_card ', 'boleto_bancario', 'otro')
    GROUP BY payment_type
    ORDER BY payment_type
""").df())

# Productos
print("PRODUCTOS")
display(con.execute("""
    SELECT
        SUM(CASE WHEN product_category_name = '  cama_mesa_banho  ' THEN 1 ELSE 0 END)
            AS categorias_con_espacios,
        SUM(CASE WHEN product_category_name = 'CAMA_MESA_BANHO' THEN 1 ELSE 0 END)
            AS categorias_mayusculas
    FROM products_dirty
""").df())

PEDIDOS
Duplicados:


,grupos_duplicados
0,50


Estados con espacios:


,registros
0,25


ARTÍCULOS


,precios_negativos,precios_cero,fletes_negativos
0,40.0,20.0,30.0


... (2 outputs left out)

PAGOS


,payment_type,cantidad
0,boleto_bancario,25
1,credit_card,40
2,otro,20


PRODUCTOS


,categorias_con_espacios,categorias_mayusculas
0,35.0,25.0


In [15]:
# LIMPIEZA

In [16]:
# 01 DUPLICADOS

# Diagnóstico de duplicados en orders
print("Duplicados en orders_dirty (por order_id):")
display(con.execute("""
    SELECT order_id, COUNT(*) AS veces
    FROM orders_dirty
    GROUP BY order_id
    HAVING COUNT(*) > 1
    ORDER BY veces DESC
    LIMIT 10
""").df())

# Diagnóstico de duplicados en order_items
print("\nDuplicados en order_items_dirty:")
display(con.execute("""
    SELECT order_id, order_item_id, COUNT(*) AS veces
    FROM order_items_dirty
    GROUP BY order_id, order_item_id
    HAVING COUNT(*) > 1
    ORDER BY veces DESC
    LIMIT 10
""").df())

# Limpieza de orders
con.execute("""
    CREATE OR REPLACE TABLE orders_clean AS
    SELECT * EXCLUDE (rn)
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY order_id
                ORDER BY order_purchase_timestamp
            ) AS rn
        FROM orders_dirty
    )
    WHERE rn = 1
""")

# Limpieza de order_items
con.execute("""
    CREATE OR REPLACE TABLE order_items_clean AS
    SELECT * EXCLUDE (rn)
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY order_id, order_item_id
                ORDER BY shipping_limit_date
            ) AS rn
        FROM order_items_dirty
    )
    WHERE rn = 1
""")

# Verificación de duplicados restantes
print("\nVerificación después de limpiar:")

display(con.execute("""
    SELECT 'orders' AS tabla, COUNT(*) AS duplicados
    FROM (
        SELECT order_id
        FROM orders_clean
        GROUP BY order_id
        HAVING COUNT(*) > 1
    )

    UNION ALL

    SELECT 'order_items', COUNT(*)
    FROM (
        SELECT order_id, order_item_id
        FROM order_items_clean
        GROUP BY order_id, order_item_id
        HAVING COUNT(*) > 1
    )
""").df())

Duplicados en orders_dirty (por order_id):


,order_id,veces
0,0008288aa423d2a3f00fcb17cd7d8719,2
1,001862358bf858722e1e2ae000cfed8b,2
2,001e7cf2ad6bef3ade12ebc56ceaf0f3,2
3,00130c0eee84a3d909e75bc08c5c3ca1,2
4,0009792311464db532ff765bf7b182ae,2
5,0015ebb40fb17286bea51d4607c4733c,2
6,00024acbcdf0a6daa1e931b038114c75,2
7,00048cc3ae777c65dbb7d2a0634bc1ea,2
8,001dbc16dc51075e987543d23a0507c7,2
9,00119ff934e539cf26f92b9ef0cdfed8,2



Duplicados en order_items_dirty:


,order_id,order_item_id,veces
0,000e906b789b55f64edcb1f84030f90d,1,2
1,000c3e6612759851cc3cbb4b83257986,1,2
2,00042b26cf59d7ce69dfabb4e55b4fd9,1,2
3,001427c0ec99cf8af737bd88e92fd444,1,2
4,0009792311464db532ff765bf7b182ae,1,2
5,00143d0f86d6fbd9f9b38ab440ac16f5,2,2
6,000aed2e25dbad2f9ddb70584c5a2ded,1,2
7,00143d0f86d6fbd9f9b38ab440ac16f5,1,2
8,0011d82c4b53e22e84023405fb467e57,1,2
9,000576fe39319847cbb9d288c5617fa6,1,2



Verificación después de limpiar:


,tabla,duplicados
0,orders,0
1,order_items,0


In [17]:

# 01. Duplicados
# Clave de negocio: order_id en orders
# y (order_id + order_item_id) en order_items
# Uso ROW_NUMBER para quedarme con una sola fila


In [18]:
# 02 VALORES NULOS

# Medir nulos en orders_clean
print("Nulos en orders_clean:")

display(con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE order_approved_at IS NULL
               OR TRIM(order_approved_at) = ''
        ) AS approved_at_nulos,

        COUNT(*) FILTER (
            WHERE order_delivered_carrier_date IS NULL
               OR TRIM(order_delivered_carrier_date) = ''
        ) AS carrier_nulos,

        COUNT(*) FILTER (
            WHERE order_delivered_customer_date IS NULL
               OR TRIM(order_delivered_customer_date) = ''
        ) AS delivered_nulos

    FROM orders_clean
""").df())


# Medir categorías nulas en products_dirty
print("\nNulos en products_dirty:")

display(con.execute("""
    SELECT
        COUNT(*) AS total_sin_categoria
    FROM products_dirty
    WHERE product_category_name IS NULL
       OR TRIM(product_category_name) = ''
""").df())


# Crear products_clean tratando categorías nulas o vacías
con.execute("""
    CREATE OR REPLACE TABLE products_clean AS
    SELECT
        * EXCLUDE (product_category_name),

        CASE
            WHEN product_category_name IS NULL
              OR TRIM(product_category_name) = ''
            THEN 'sin_categoria'
            ELSE product_category_name
        END AS product_category_name

    FROM products_dirty
""")

print("products_clean creada correctamente.")

Nulos en orders_clean:


,approved_at_nulos,carrier_nulos,delivered_nulos
0,160,1783,2964



Nulos en products_dirty:


,total_sin_categoria
0,609


products_clean creada correctamente.


In [19]:
display(con.execute("""
    SELECT COUNT(*) AS categorias_sin_clasificar
    FROM products_clean
    WHERE product_category_name IS NULL
       OR TRIM(product_category_name) = ''
""").df())

,categorias_sin_clasificar
0,0


In [20]:
# 02. Valores nulos
# Mido nulos críticos.
# Decisión: fechas nulas se conservan (son normales).
# Categoría nula se imputa como 'sin_categoria'.

In [21]:
# 03 ESTANDARIZACIÓN DE TEXTO

# Clientes
con.execute("""
    CREATE OR REPLACE TABLE customers_clean AS
    SELECT
        customer_id,
        customer_unique_id,
        customer_zip_code_prefix,
        LOWER(TRIM(customer_city)) AS customer_city,
        UPPER(TRIM(customer_state)) AS customer_state
    FROM customers_dirty
""")

# Pedidos
con.execute("""
    CREATE OR REPLACE TABLE orders_clean AS
    SELECT
        * REPLACE (
            LOWER(TRIM(order_status)) AS order_status
        )
    FROM orders_clean
""")

# Productos
con.execute("""
    CREATE OR REPLACE TABLE products_clean AS
    SELECT
        * REPLACE (
            LOWER(TRIM(product_category_name)) AS product_category_name
        )
    FROM products_clean
""")

# Pagos
con.execute("""
    CREATE OR REPLACE TABLE payments_clean AS
    SELECT
        order_id,
        payment_sequential,
        LOWER(TRIM(payment_type)) AS payment_type,
        payment_installments,
        payment_value
    FROM payments_dirty
""")

print("Texto estandarizado.")

Texto estandarizado.


In [22]:
# VALIDACIÓN
# Verificar espacios sobrantes en ciudades
display(con.execute("""
    SELECT COUNT(*) AS ciudades_con_espacios
    FROM customers_clean
    WHERE customer_city <> TRIM(customer_city)
""").df())

# Verificar espacios sobrantes en estados de pedidos
display(con.execute("""
    SELECT COUNT(*) AS estados_con_espacios
    FROM orders_clean
    WHERE order_status <> TRIM(order_status)
""").df())

# Verificar tipos de pago
display(con.execute("""
    SELECT payment_type, COUNT(*) AS cantidad
    FROM payments_clean
    GROUP BY payment_type
    ORDER BY cantidad DESC
""").df())

,ciudades_con_espacios
0,0


,estados_con_espacios
0,0


,payment_type,cantidad
0,credit_card,76775
1,boleto,19763
2,voucher,5772
3,debit_card,1528
4,boleto_bancario,25
5,otro,20
6,not_defined,3


In [23]:
# 03. Estandarización de texto
# Quito espacios y normalizo mayúsculas/minúsculas
# en ciudades, estados, status y categorías.

In [24]:
# 04 TIPOS DE DATOS

# Pedidos: convertir fechas a TIMESTAMP
con.execute("""
    CREATE OR REPLACE TABLE orders_clean AS
    SELECT
        order_id,
        customer_id,
        order_status,
        TRY_CAST(order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
        TRY_CAST(order_approved_at AS TIMESTAMP) AS order_approved_at,
        TRY_CAST(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_date,
        TRY_CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
        TRY_CAST(order_estimated_delivery_date AS TIMESTAMP) AS order_estimated_delivery_date
    FROM orders_clean
""")

# Artículos: convertir fechas y valores numéricos
con.execute("""
    CREATE OR REPLACE TABLE order_items_clean AS
    SELECT
        order_id,
        TRY_CAST(order_item_id AS INTEGER) AS order_item_id,
        product_id,
        seller_id,
        TRY_CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date,
        TRY_CAST(price AS DOUBLE) AS price,
        TRY_CAST(freight_value AS DOUBLE) AS freight_value
    FROM order_items_clean
""")

# Pagos: convertir valores numéricos
con.execute("""
    CREATE OR REPLACE TABLE payments_clean AS
    SELECT
        order_id,
        TRY_CAST(payment_sequential AS INTEGER) AS payment_sequential,
        payment_type,
        TRY_CAST(payment_installments AS INTEGER) AS payment_installments,
        TRY_CAST(payment_value AS DOUBLE) AS payment_value
    FROM payments_clean
""")

# Productos: convertir columnas numéricas
con.execute("""
    CREATE OR REPLACE TABLE products_clean AS
    SELECT
        product_id,
        product_category_name,
        TRY_CAST(product_name_lenght AS INTEGER) AS product_name_lenght,
        TRY_CAST(product_description_lenght AS INTEGER) AS product_description_lenght,
        TRY_CAST(product_photos_qty AS INTEGER) AS product_photos_qty,
        TRY_CAST(product_weight_g AS DOUBLE) AS product_weight_g,
        TRY_CAST(product_length_cm AS DOUBLE) AS product_length_cm,
        TRY_CAST(product_height_cm AS DOUBLE) AS product_height_cm,
        TRY_CAST(product_width_cm AS DOUBLE) AS product_width_cm
    FROM products_clean
""")

print("Tipos de datos convertidos.")

Tipos de datos convertidos.


In [25]:
# 04. Tipos de datos
# Paso texto a TIMESTAMP, INTEGER y DOUBLE
# con TRY_CAST para manejar valores inválidos.

In [26]:
# Revisar fechas de pedidos que no pudieron convertirse
display(con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE order_purchase_timestamp IS NULL
        ) AS purchase_nulos,

        COUNT(*) FILTER (
            WHERE order_approved_at IS NULL
        ) AS approved_nulos

    FROM orders_clean
""").df())

# Revisar tipos de datos resultantes
display(con.execute("""
    DESCRIBE orders_clean
""").df())

display(con.execute("""
    DESCRIBE order_items_clean
""").df())

,purchase_nulos,approved_nulos
0,0,160


,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_purchase_timestamp,TIMESTAMP,YES,None,None,None
4,order_approved_at,TIMESTAMP,YES,None,None,None
5,order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
6,order_delivered_customer_date,TIMESTAMP,YES,None,None,None
7,order_estimated_delivery_date,TIMESTAMP,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,order_item_id,INTEGER,YES,None,None,None
2,product_id,VARCHAR,YES,None,None,None
3,seller_id,VARCHAR,YES,None,None,None
4,shipping_limit_date,TIMESTAMP,YES,None,None,None
5,price,DOUBLE,YES,None,None,None
6,freight_value,DOUBLE,YES,None,None,None


In [27]:
# 05 FECHAS INCONSISTENTES

print("Entregas anteriores a la compra:")

display(con.execute("""
    SELECT
        order_id,
        order_purchase_timestamp,
        order_delivered_customer_date
    FROM orders_clean
    WHERE order_delivered_customer_date < order_purchase_timestamp
    LIMIT 10
""").df())

print("Entregas al transportista anteriores a la compra:")

display(con.execute("""
    SELECT
        order_id,
        order_purchase_timestamp,
        order_delivered_carrier_date
    FROM orders_clean
    WHERE order_delivered_carrier_date < order_purchase_timestamp
    LIMIT 10
""").df())

print("Entregas al cliente anteriores a la entrega al transportista:")

display(con.execute("""
    SELECT
        order_id,
        order_delivered_carrier_date,
        order_delivered_customer_date
    FROM orders_clean
    WHERE order_delivered_customer_date < order_delivered_carrier_date
    LIMIT 10
""").df())

Entregas anteriores a la compra:


,order_id,order_purchase_timestamp,order_delivered_customer_date
0,00125cb692d04887809806618a2a145f,2017-03-23 12:21:17,2016-01-15 10:00:00
1,001c85b5f68d2be0cb0797afc9e8ce9a,2017-11-24 19:19:18,2016-01-15 10:00:00
2,00042b26cf59d7ce69dfabb4e55b4fd9,2017-02-04 13:57:51,2016-01-15 10:00:00
3,000229ec398224ef6ca0657da4fc703e,2018-01-14 14:33:31,2016-01-15 10:00:00
4,0020262c8a370bd5a174ea6a2a267321,2017-11-28 09:32:49,2016-01-15 10:00:00
5,0015ebb40fb17286bea51d4607c4733c,2018-01-14 09:01:36,2016-01-15 10:00:00
6,002955b0acc63d04f6724a31967e82bd,2017-12-18 13:43:54,2016-01-15 10:00:00
7,0029c5db35ef291ce99d1262012e7daa,2018-08-25 00:09:40,2016-01-15 10:00:00
8,001e7cf2ad6bef3ade12ebc56ceaf0f3,2018-05-19 10:29:23,2016-01-15 10:00:00
9,0010b2e5201cc5f1ae7e9c6cc8f5bd00,2017-09-11 17:39:33,2016-01-15 10:00:00


Entregas al transportista anteriores a la compra:


,order_id,order_purchase_timestamp,order_delivered_carrier_date
0,b9afddbdcfadc9a87b41a83271c3e888,2018-08-16 13:50:48,2018-08-16 13:27:00
1,ad133696906f6a78826daa0911b7daec,2018-06-15 15:41:22,2018-06-15 14:52:00
2,74e033208dc13a7b8127eb8e73d09b76,2018-05-02 10:48:44,2018-05-02 09:49:00
3,a6b58794fd2ba533359a76c08df576e3,2018-05-14 15:18:23,2018-05-14 13:46:00
4,5792e0b1c8c8a2bf53af468c9a422c88,2018-07-26 13:25:14,2018-07-26 12:42:00
5,c3eb293fd154223498b6551a728203e8,2018-07-19 14:06:04,2018-07-19 13:49:00
6,b0c2a7d04b165525254254a728c50a4e,2018-06-07 13:28:30,2018-06-07 13:22:00
7,2033a4586b5bec3229ebc1675a8ae092,2018-06-12 10:10:25,2018-06-12 10:09:00
8,08adcddad19d3acf37d1fa01cb9ded1e,2018-06-27 11:16:44,2018-06-27 10:57:00
9,dee6298ce7d1fb2645141ef9972157aa,2018-04-30 14:06:12,2018-04-30 12:59:00


Entregas al cliente anteriores a la entrega al transportista:


,order_id,order_delivered_carrier_date,order_delivered_customer_date
0,00125cb692d04887809806618a2a145f,2017-03-27 08:58:04,2016-01-15 10:00:00
1,001c85b5f68d2be0cb0797afc9e8ce9a,2017-11-27 12:42:15,2016-01-15 10:00:00
2,00042b26cf59d7ce69dfabb4e55b4fd9,2017-02-16 09:46:09,2016-01-15 10:00:00
3,000229ec398224ef6ca0657da4fc703e,2018-01-16 12:36:48,2016-01-15 10:00:00
4,a1abeb653a4d4cd1e142ccb8c82cd069,2017-07-28 16:57:58,2017-07-25 19:32:56
5,0020262c8a370bd5a174ea6a2a267321,2017-11-29 00:09:20,2016-01-15 10:00:00
6,0015ebb40fb17286bea51d4607c4733c,2018-01-17 22:56:36,2016-01-15 10:00:00
7,002955b0acc63d04f6724a31967e82bd,2017-12-19 17:51:40,2016-01-15 10:00:00
8,383aa8b2724fe452d9ccd9934a8c628b,2017-07-07 17:22:41,2017-07-06 14:27:51
9,0029c5db35ef291ce99d1262012e7daa,2018-08-29 13:24:00,2016-01-15 10:00:00


In [28]:
con.execute("""
    CREATE OR REPLACE TABLE orders_clean AS
    SELECT
        * REPLACE (
            CASE
                WHEN order_delivered_carrier_date < order_purchase_timestamp
                THEN NULL
                ELSE order_delivered_carrier_date
            END AS order_delivered_carrier_date,

            CASE
                WHEN order_delivered_customer_date < order_purchase_timestamp
                  OR order_delivered_customer_date < order_delivered_carrier_date
                THEN NULL
                ELSE order_delivered_customer_date
            END AS order_delivered_customer_date
        )
    FROM orders_clean
""")

print("Fechas inconsistentes tratadas.")

Fechas inconsistentes tratadas.


In [29]:
display(con.execute("""
    SELECT
        order_id,
        order_status,
        order_purchase_timestamp,
        order_delivered_carrier_date,
        order_delivered_customer_date,
        CASE
            WHEN order_delivered_customer_date < order_purchase_timestamp
                THEN 'cliente_antes_compra'
            WHEN order_delivered_customer_date < order_delivered_carrier_date
                THEN 'cliente_antes_carrier'
        END AS inconsistencia
    FROM orders_clean
    WHERE
        order_delivered_customer_date < order_purchase_timestamp
        OR order_delivered_customer_date < order_delivered_carrier_date
    ORDER BY order_purchase_timestamp
    LIMIT 20
""").df())

,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,inconsistencia


In [30]:
# 05. Fechas inconsistentes
# Detecto entregas anteriores a la compra
# y otras secuencias imposibles.
# Las fechas incoherentes las paso a NULL.

In [31]:
# 06 PRECIOS INVALIDOS

print("Precios negativos o iguales a cero:")

display(con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE price < 0) AS precios_negativos,
        COUNT(*) FILTER (WHERE price = 0) AS precios_cero,
        COUNT(*) FILTER (WHERE freight_value < 0) AS fletes_negativos
    FROM order_items_clean
""").df())

Precios negativos o iguales a cero:


,precios_negativos,precios_cero,fletes_negativos
0,40,20,30


In [32]:
con.execute("""
    CREATE OR REPLACE TABLE order_items_clean AS
    SELECT
        * REPLACE (
            CASE
                WHEN price <= 0 THEN NULL
                ELSE price
            END AS price,

            CASE
                WHEN freight_value < 0 THEN NULL
                ELSE freight_value
            END AS freight_value
        )
    FROM order_items_clean
""")

print("Precios y fletes inválidos tratados.")

Precios y fletes inválidos tratados.


In [33]:
# 06. Precios inválidos
# Política: precio <= 0 o flete < 0 → NULL
# No elimino el registro para no perder el resto de la info.

In [34]:
# 07 VALORES ATIPICOS 


print("Límites IQR para precio:")

display(con.execute("""
    WITH stats AS (
        SELECT
            QUANTILE_CONT(price, 0.25) AS q1,
            QUANTILE_CONT(price, 0.75) AS q3
        FROM order_items_clean
        WHERE price IS NOT NULL
    )

    SELECT
        q1,
        q3,
        q3 - q1 AS iqr,
        q1 - 1.5 * (q3 - q1) AS limite_inferior,
        q3 + 1.5 * (q3 - q1) AS limite_superior
    FROM stats
""").df())

Límites IQR para precio:


,q1,q3,iqr,limite_inferior,limite_superior
0,39.9,134.9,95.0,-102.6,277.4


In [35]:
print("Cantidad de valores atípicos en precios:")

display(con.execute("""
    WITH stats AS (
        SELECT
            QUANTILE_CONT(price, 0.25) AS q1,
            QUANTILE_CONT(price, 0.75) AS q3
        FROM order_items_clean
        WHERE price IS NOT NULL
    )

    SELECT
        COUNT(*) AS outliers_precio
    FROM order_items_clean, stats
    WHERE price < q1 - 1.5 * (q3 - q1)
       OR price > q3 + 1.5 * (q3 - q1)
""").df())

Cantidad de valores atípicos en precios:


,outliers_precio
0,8422


In [36]:
# Límites IQR para flete

print("Límites IQR para flete:")

display(con.execute("""
    WITH stats AS (
        SELECT
            QUANTILE_CONT(freight_value, 0.25) AS q1,
            QUANTILE_CONT(freight_value, 0.75) AS q3
        FROM order_items_clean
        WHERE freight_value IS NOT NULL
    )
    SELECT
        q1,
        q3,
        q3 - q1 AS iqr,
        q1 - 1.5 * (q3 - q1) AS limite_inferior,
        q3 + 1.5 * (q3 - q1) AS limite_superior
    FROM stats
""").df())

print("Cantidad de valores atípicos en fletes:")

display(con.execute("""
    WITH stats AS (
        SELECT
            QUANTILE_CONT(freight_value, 0.25) AS q1,
            QUANTILE_CONT(freight_value, 0.75) AS q3
        FROM order_items_clean
        WHERE freight_value IS NOT NULL
    )
    SELECT
        COUNT(*) AS outliers_flete
    FROM order_items_clean, stats
    WHERE freight_value < q1 - 1.5 * (q3 - q1)
       OR freight_value > q3 + 1.5 * (q3 - q1)
""").df())

Límites IQR para flete:


,q1,q3,iqr,limite_inferior,limite_superior
0,13.08,21.15,8.07,0.975,33.255


Cantidad de valores atípicos en fletes:


,outliers_flete
0,12130


In [37]:
# Límites IQR para pagos

print("Límites IQR para pagos:")

display(con.execute("""
    WITH stats AS (
        SELECT
            QUANTILE_CONT(payment_value, 0.25) AS q1,
            QUANTILE_CONT(payment_value, 0.75) AS q3
        FROM payments_clean
        WHERE payment_value IS NOT NULL
    )
    SELECT
        q1,
        q3,
        q3 - q1 AS iqr,
        q1 - 1.5 * (q3 - q1) AS limite_inferior,
        q3 + 1.5 * (q3 - q1) AS limite_superior
    FROM stats
""").df())

print("Cantidad de valores atípicos en pagos:")

display(con.execute("""
    WITH stats AS (
        SELECT
            QUANTILE_CONT(payment_value, 0.25) AS q1,
            QUANTILE_CONT(payment_value, 0.75) AS q3
        FROM payments_clean
        WHERE payment_value IS NOT NULL
    )
    SELECT
        COUNT(*) AS outliers_pago
    FROM payments_clean, stats
    WHERE payment_value < q1 - 1.5 * (q3 - q1)
       OR payment_value > q3 + 1.5 * (q3 - q1)
""").df())

Límites IQR para pagos:


,q1,q3,iqr,limite_inferior,limite_superior
0,56.79,171.8375,115.0475,-115.78125,344.40875


Cantidad de valores atípicos en pagos:


,outliers_pago
0,7981


In [38]:
#Se utilizó el método del rango intercuartílico (IQR) para identificar valores atípicos en los precios de los productos, los costos de envío y los montos de pago. 
#Se identificaron 8,422 valores atípicos en precios, 12,130 en fletes y 7,981 en pagos. 
#Estos registros se conservaron, ya que los valores extremos pueden representar operaciones comerciales legítimas y no necesariamente errores. 
#Se utilizaron los límites calculados como referencia para el análisis de calidad de los datos.


In [39]:
# 08 CATEGORIAS

display(con.execute("""
    SELECT
        product_category_name,
        COUNT(*) AS cantidad
    FROM products_clean
    GROUP BY product_category_name
    ORDER BY product_category_name
""").df())

,product_category_name,cantidad
0,agro_industria_e_comercio,74
1,alimentos,82
2,alimentos_bebidas,104
3,artes,55
4,artes_e_artesanato,19
...,...,...
69,sinalizacao_e_seguranca,93
70,tablets_impressao_imagem,9
71,telefonia,1134
72,telefonia_fixa,116


In [40]:
con.execute("""
    CREATE OR REPLACE TABLE products_clean AS
    SELECT * REPLACE (
        CASE
            WHEN product_category_name IN (
                'cama_mesa_banho', 'cama mesa banho'
            ) THEN 'cama_mesa_banho'

            WHEN product_category_name IN (
                'esporte_lazer', 'esporte lazer'
            ) THEN 'esporte_lazer'

            WHEN product_category_name IN (
                'moveis_decoracao', 'moveis decoracao'
            ) THEN 'moveis_decoracao'

            ELSE product_category_name
        END AS product_category_name
    )
    FROM products_clean
""")

In [41]:
display(con.execute("""
    SELECT
        product_category_name,
        COUNT(*) AS cantidad
    FROM products_clean
    WHERE product_category_name LIKE '% %'
       OR product_category_name LIKE '%  %'
    GROUP BY product_category_name
    ORDER BY product_category_name
""").df())

,product_category_name,cantidad


In [42]:
# 08. Categorías
# Unifico nombres equivalentes con un CASE
# (catálogo simple de correspondencias).

In [43]:
# 09 INTEGRIDAD REFERENCIAL

display(con.execute("""
    SELECT 'items_sin_pedido' AS relacion, COUNT(*) AS huerfanos
    FROM order_items_clean oi
    LEFT JOIN orders_clean o
        ON oi.order_id = o.order_id
    WHERE o.order_id IS NULL

    UNION ALL

    SELECT 'pagos_sin_pedido', COUNT(*)
    FROM payments_clean p
    LEFT JOIN orders_clean o
        ON p.order_id = o.order_id
    WHERE o.order_id IS NULL

    UNION ALL

    SELECT 'pedidos_sin_cliente', COUNT(*)
    FROM orders_clean o
    LEFT JOIN customers_clean c
        ON o.customer_id = c.customer_id
    WHERE c.customer_id IS NULL

    UNION ALL

    SELECT 'items_sin_producto', COUNT(*)
    FROM order_items_clean oi
    LEFT JOIN products_clean p
        ON oi.product_id = p.product_id
    WHERE p.product_id IS NULL
""").df())

,relacion,huerfanos
0,items_sin_pedido,0
1,pagos_sin_pedido,0
2,pedidos_sin_cliente,0
3,items_sin_producto,0


In [44]:
# 09. Integridad referencial
# Busco registros huérfanos con LEFT JOIN.

In [45]:
# 10 ESTADO DE PEDIDO

display(con.execute("""
    SELECT
        order_status,
        COUNT(*) AS cantidad,
        COUNT(*) FILTER (
            WHERE order_delivered_carrier_date IS NOT NULL
        ) AS con_fecha_transportista,
        COUNT(*) FILTER (
            WHERE order_delivered_customer_date IS NOT NULL
        ) AS con_fecha_entrega_cliente
    FROM orders_clean
    GROUP BY order_status
    ORDER BY order_status
""").df())

,order_status,cantidad,con_fecha_transportista,con_fecha_entrega_cliente
0,approved,2,0,0
1,canceled,660,110,41
2,created,5,0,0
3,delivered,96443,96276,96343
4,invoiced,314,0,0
5,processing,301,0,0
6,shipped,1107,1106,0
7,unavailable,609,0,0


In [46]:
# PEDIDOS CANCELADOS CON FECHA DE ENTREGA
display(con.execute("""
    SELECT
        order_id,
        order_status,
        order_delivered_carrier_date,
        order_delivered_customer_date
    FROM orders_clean
    WHERE order_status = 'canceled'
      AND (
          order_delivered_carrier_date IS NOT NULL
          OR order_delivered_customer_date IS NOT NULL
      )
    LIMIT 20
""").df())

,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date
0,714fb133a6730ab81fa1d3c1b2007291,canceled,2018-01-29 22:33:25,NaT
1,3a129877493c8189c59c60eb71d97c29,canceled,2018-01-26 21:42:18,NaT
2,002f16b7bc4530031b7d90f791b12d8a,canceled,2018-07-03 16:35:00,2018-07-06 18:14:47
3,1950d777989f6a877539f53795b4c3c3,canceled,2018-02-20 19:57:13,2018-03-21 22:03:51
4,00345f338696283410b7977d2e3efc89,canceled,2018-08-21 13:18:00,2018-08-30 21:13:48
5,8a9cce19aa49e055f2babb81d5a71c40,canceled,2018-02-05 20:41:28,NaT
6,c1049eef89915c6beb3604bf3d6588d7,canceled,2018-02-09 22:13:01,NaT
7,885935156ba43094f3ae95f3c14e0381,canceled,2018-02-07 20:48:31,NaT
8,5b4858f2dcf69731f2285a38829bfa7c,canceled,2018-02-16 16:52:52,NaT
9,004157daafa0bc8672e01f00e4f0c04f,canceled,2017-05-25 12:05:10,2017-06-02 14:57:49


In [47]:
con.execute("""
    CREATE OR REPLACE TABLE orders_clean AS
    SELECT * REPLACE (
        CASE
            WHEN order_status = 'canceled'
             AND order_delivered_customer_date IS NOT NULL
            THEN 'delivered'
            ELSE order_status
        END AS order_status
    )
    FROM orders_clean
""")

In [48]:
display(con.execute("""
    SELECT
        o.order_id,
        o.order_status AS estado_actual,
        r.order_status AS estado_original,
        o.order_delivered_carrier_date,
        o.order_delivered_customer_date
    FROM orders_clean o
    INNER JOIN orders_raw r
        ON o.order_id = r.order_id
    WHERE o.order_status = 'canceled'
      AND o.order_delivered_customer_date IS NOT NULL
    ORDER BY o.order_id
    LIMIT 50
""").df())

,order_id,estado_actual,estado_original,order_delivered_carrier_date,order_delivered_customer_date


In [49]:
display(con.execute("""
    SELECT
        order_status,
        COUNT(*) AS cantidad,
        COUNT(*) FILTER (
            WHERE order_delivered_customer_date IS NOT NULL
        ) AS con_fecha_entrega_cliente
    FROM orders_clean
    GROUP BY order_status
    ORDER BY order_status
""").df())

,order_status,cantidad,con_fecha_entrega_cliente
0,approved,2,0
1,canceled,619,0
2,created,5,0
3,delivered,96484,96384
4,invoiced,314,0
5,processing,301,0
6,shipped,1107,0
7,unavailable,609,0


In [50]:
# 10. Estados de pedido
# Detecto canceled con fecha de entrega.
# Si tiene fecha de entrega, lo dejo como delivered.

In [51]:
# 11 CODIGOS POSTALES
display(con.execute("""
    SELECT
        LENGTH(customer_zip_code_prefix) AS longitud,
        COUNT(*) AS cantidad
    FROM customers_clean
    GROUP BY LENGTH(customer_zip_code_prefix)
    ORDER BY longitud
""").df())


,longitud,cantidad
0,3,25
1,5,99416


In [52]:
con.execute("""
    CREATE OR REPLACE TABLE customers_clean AS
    SELECT * REPLACE (
        CASE
            WHEN REGEXP_MATCHES(
                customer_zip_code_prefix,
                '^[0-9]{1,5}$'
            )
            THEN LPAD(customer_zip_code_prefix, 5, '0')
            ELSE NULL
        END AS customer_zip_code_prefix
    )
    FROM customers_clean
""")

print("Códigos postales validados y normalizados.")

Códigos postales validados y normalizados.


In [53]:
# 11. Códigos postales
# Valido que sean numéricos de hasta 5 dígitos
# y relleno con ceros a la izquierda (LPAD).

In [54]:
# 12 PAGOS 
# COMPARAR TOTALES POR PEDIDOS
display(con.execute("""
WITH totales_items AS (
    SELECT
        order_id,
        SUM(price + freight_value) AS total_esperado,
        COUNT(*) AS cantidad_items,
        COUNT(*) FILTER (
            WHERE price IS NULL OR freight_value IS NULL
        ) AS items_con_importes_nulos
    FROM order_items_clean
    GROUP BY order_id
),
totales_pagos AS (
    SELECT
        order_id,
        SUM(payment_value) AS total_pagado,
        COUNT(*) AS cantidad_pagos
    FROM payments_clean
    GROUP BY order_id
)
SELECT
    i.order_id,
    i.total_esperado,
    p.total_pagado,
    ROUND(ABS(i.total_esperado - p.total_pagado), 2) AS diferencia,
    i.cantidad_items,
    p.cantidad_pagos,
    i.items_con_importes_nulos
FROM totales_items i
INNER JOIN totales_pagos p
    ON i.order_id = p.order_id
WHERE i.total_esperado IS NOT NULL
  AND i.items_con_importes_nulos = 0
  AND ABS(i.total_esperado - p.total_pagado) > 0.01
ORDER BY diferencia DESC
LIMIT 30
""").df())

,order_id,total_esperado,total_pagado,diferencia,cantidad_items,cantidad_pagos,items_con_importes_nulos
0,ce6d150fb29ada17d2082f4847107665,1403.66,1586.47,182.81,1,1,0
1,6e5fe7366a2e1bfbf3257dba0af1267f,287.91,406.92,119.01,6,1,0
2,70b742795bc441e94a44a084b6d9ce7a,466.93,578.82,111.89,1,1,0
3,996c7e73600ad3723e8627ab7bef81e4,587.90,664.43,76.53,1,1,0
4,70b7e94ea46d3e8b5bc12a50186edaf0,213.15,274.84,61.69,3,1,0
5,bc2c82b0ef78d2252b6176d1972db7c9,242.01,303.02,61.01,3,1,0
6,af9ffff2ce6b3defd34fd4c78857a379,413.17,466.97,53.80,1,1,0
7,262118ce178bb3e4590a3adcf6d62e6b,177.74,126.12,51.62,2,1,0
8,bfdb5bbb06458d600a33d61f5f287472,348.93,394.36,45.43,1,1,0
9,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,254.45,293.89,39.44,2,1,0


In [55]:
# RESUMEN DE COINCIDENCIAS Y DIFERENCIAS
display(con.execute("""
WITH totales_items AS (
    SELECT
        order_id,
        SUM(price + freight_value) AS total_esperado,
        COUNT(*) FILTER (
            WHERE price IS NULL OR freight_value IS NULL
        ) AS importes_nulos
    FROM order_items_clean
    GROUP BY order_id
),
totales_pagos AS (
    SELECT
        order_id,
        SUM(payment_value) AS total_pagado
    FROM payments_clean
    GROUP BY order_id
),
comparacion AS (
    SELECT
        i.order_id,
        i.total_esperado,
        p.total_pagado,
        i.importes_nulos,
        ABS(i.total_esperado - p.total_pagado) AS diferencia
    FROM totales_items i
    INNER JOIN totales_pagos p USING (order_id)
)
SELECT
    COUNT(*) AS pedidos_comparados,
    COUNT(*) FILTER (
        WHERE importes_nulos = 0
          AND diferencia <= 0.01
    ) AS montos_coincidentes,
    COUNT(*) FILTER (
        WHERE importes_nulos = 0
          AND diferencia > 0.01
    ) AS montos_diferentes,
    COUNT(*) FILTER (
        WHERE importes_nulos > 0
    ) AS pedidos_con_importes_incompletos
FROM comparacion
""").df())

,pedidos_comparados,montos_coincidentes,montos_diferentes,pedidos_con_importes_incompletos
0,98665,98204,380,81


In [56]:
# RESUMEN DE DIFERENCIA
display(con.execute("""
WITH totales_items AS (
    SELECT
        order_id,
        SUM(price + freight_value) AS total_esperado,
        COUNT(*) FILTER (
            WHERE price IS NULL OR freight_value IS NULL
        ) AS importes_nulos
    FROM order_items_clean
    GROUP BY order_id
),
totales_pagos AS (
    SELECT
        order_id,
        SUM(payment_value) AS total_pagado
    FROM payments_clean
    GROUP BY order_id
),
comparacion AS (
    SELECT
        i.order_id,
        i.total_esperado,
        p.total_pagado,
        i.importes_nulos,
        ABS(i.total_esperado - p.total_pagado) AS diferencia
    FROM totales_items i
    INNER JOIN totales_pagos p USING (order_id)
)
SELECT
    COUNT(*) AS pedidos_comparados,
    COUNT(*) FILTER (
        WHERE importes_nulos = 0
          AND diferencia <= 0.01
    ) AS montos_coincidentes,
    COUNT(*) FILTER (
        WHERE importes_nulos = 0
          AND diferencia > 0.01
    ) AS montos_diferentes,
    COUNT(*) FILTER (
        WHERE importes_nulos > 0
    ) AS pedidos_con_importes_incompletos
FROM comparacion
""").df())

,pedidos_comparados,montos_coincidentes,montos_diferentes,pedidos_con_importes_incompletos
0,98665,98204,380,81


In [57]:
# 12. Pagos
# Comparo suma de (precio + flete) vs suma de pagos
# por pedido. Considero pagos múltiples.

In [58]:
# 13 METODOS DE PAGO


con.execute("""
CREATE OR REPLACE TABLE payments_clean AS
SELECT
    order_id,
    payment_sequential,
    CASE
        WHEN TRIM(LOWER(payment_type)) = 'credit_card'
            THEN 'credit_card'
        WHEN TRIM(LOWER(payment_type)) IN ('boleto', 'boleto_bancario')
            THEN 'boleto'
        WHEN TRIM(LOWER(payment_type)) = 'voucher'
            THEN 'voucher'
        WHEN TRIM(LOWER(payment_type)) = 'debit_card'
            THEN 'debit_card'
        WHEN TRIM(LOWER(payment_type)) = 'not_defined'
            THEN 'not_defined'
        WHEN TRIM(LOWER(payment_type)) = 'otro'
            THEN 'otro'
        ELSE 'desconocido'
    END AS payment_type,
    payment_installments,
    payment_value
FROM payments_clean
""")

print("Métodos de pago normalizados")

Métodos de pago normalizados


In [59]:
display(con.execute("""
SELECT
    payment_type,
    COUNT(*) AS cantidad
FROM payments_clean
GROUP BY payment_type
ORDER BY cantidad DESC
""").df())

,payment_type,cantidad
0,credit_card,76775
1,boleto,19788
2,voucher,5772
3,debit_card,1528
4,otro,20
5,not_defined,3


In [60]:
display(con.execute("""
SELECT
    payment_type,
    COUNT(*) AS cantidad
FROM payments_dirty
GROUP BY payment_type
ORDER BY cantidad DESC
""").df())

,payment_type,cantidad
0,credit_card,76735
1,boleto,19763
2,voucher,5772
3,debit_card,1528
4,credit_card,40
5,boleto_bancario,25
6,otro,20
7,not_defined,3


In [61]:
# 13. Métodos de pago
# Normalizo nombres y marco los desconocidos.

In [62]:
# 14 CADENAS VACIAS

display(con.execute("""
SELECT
    'customers_clean' AS tabla,
    'customer_zip_code_prefix' AS columna,
    COUNT(*) FILTER (
        WHERE TRIM(customer_zip_code_prefix) = ''
    ) AS cadenas_vacias
FROM customers_clean

UNION ALL

SELECT
    'customers_clean',
    'customer_city',
    COUNT(*) FILTER (
        WHERE TRIM(customer_city) = ''
    )
FROM customers_clean

UNION ALL

SELECT
    'customers_clean',
    'customer_state',
    COUNT(*) FILTER (
        WHERE TRIM(customer_state) = ''
    )
FROM customers_clean

UNION ALL

SELECT
    'products_clean',
    'product_category_name',
    COUNT(*) FILTER (
        WHERE TRIM(product_category_name) = ''
    )
FROM products_clean

UNION ALL

SELECT
    'payments_clean',
    'payment_type',
    COUNT(*) FILTER (
        WHERE TRIM(payment_type) = ''
    )
FROM payments_clean

UNION ALL

SELECT
    'orders_clean',
    'order_status',
    COUNT(*) FILTER (
        WHERE TRIM(order_status) = ''
    )
FROM orders_clean
""").df())

,tabla,columna,cadenas_vacias
0,customers_clean,customer_zip_code_prefix,0
1,customers_clean,customer_city,0
2,customers_clean,customer_state,0
3,products_clean,product_category_name,0
4,payments_clean,payment_type,0
5,orders_clean,order_status,0


In [63]:
# 14. Cadenas vacías


display(con.execute("""
SELECT
    'customers_clean' AS tabla,
    'customer_zip_code_prefix' AS columna,
    COUNT(*) FILTER (WHERE TRIM(customer_zip_code_prefix) = '') AS cadenas_vacias
FROM customers_clean

UNION ALL

SELECT 'customers_clean', 'customer_city',
    COUNT(*) FILTER (WHERE TRIM(customer_city) = '')
FROM customers_clean

UNION ALL

SELECT 'customers_clean', 'customer_state',
    COUNT(*) FILTER (WHERE TRIM(customer_state) = '')
FROM customers_clean

UNION ALL

SELECT 'products_clean', 'product_category_name',
    COUNT(*) FILTER (WHERE TRIM(product_category_name) = '')
FROM products_clean

UNION ALL

SELECT 'payments_clean', 'payment_type',
    COUNT(*) FILTER (WHERE TRIM(payment_type) = '')
FROM payments_clean

UNION ALL

SELECT 'orders_clean', 'order_status',
    COUNT(*) FILTER (WHERE TRIM(order_status) = '')
FROM orders_clean
""").df())


# Ahora sí las convierto a NULL
con.execute("""
CREATE OR REPLACE TABLE customers_clean AS
SELECT
    customer_id,
    customer_unique_id,
    NULLIF(TRIM(customer_zip_code_prefix), '') AS customer_zip_code_prefix,
    NULLIF(TRIM(customer_city), '') AS customer_city,
    NULLIF(TRIM(customer_state), '') AS customer_state
FROM customers_clean
""")

con.execute("""
CREATE OR REPLACE TABLE products_clean AS
SELECT
    product_id,
    NULLIF(TRIM(product_category_name), '') AS product_category_name,
    product_name_lenght,
    product_description_lenght,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
FROM products_clean
""")

print("Cadenas vacías convertidas a NULL")

,tabla,columna,cadenas_vacias
0,customers_clean,customer_zip_code_prefix,0
1,customers_clean,customer_city,0
2,customers_clean,customer_state,0
3,products_clean,product_category_name,0
4,payments_clean,payment_type,0
5,orders_clean,order_status,0


Cadenas vacías convertidas a NULL


In [64]:
# 14. Cadenas vacías
# Diferencio NULL, cadena vacía y solo espacios.
# Convierto vacíos a NULL con NULLIF + TRIM.

In [65]:
# 15 VALIDACIÓN FINAL


print("Unicidad y valores nulos en claves:")

display(con.execute("""
SELECT
    'orders_clean' AS tabla,
    COUNT(*) - COUNT(DISTINCT order_id) AS claves_duplicadas,
    COUNT(*) FILTER (WHERE order_id IS NULL) AS claves_nulas
FROM orders_clean

UNION ALL

SELECT
    'order_items_clean',
    COUNT(*) - COUNT(DISTINCT (order_id, order_item_id)),
    COUNT(*) FILTER (
        WHERE order_id IS NULL OR order_item_id IS NULL
    )
FROM order_items_clean

UNION ALL

SELECT
    'customers_clean',
    COUNT(*) - COUNT(DISTINCT customer_id),
    COUNT(*) FILTER (WHERE customer_id IS NULL)
FROM customers_clean

UNION ALL

SELECT
    'products_clean',
    COUNT(*) - COUNT(DISTINCT product_id),
    COUNT(*) FILTER (WHERE product_id IS NULL)
FROM products_clean

UNION ALL

SELECT
    'payments_clean',
    COUNT(*) - COUNT(DISTINCT (order_id, payment_sequential)),
    COUNT(*) FILTER (
        WHERE order_id IS NULL OR payment_sequential IS NULL
    )
FROM payments_clean
""").df())

Unicidad y valores nulos en claves:


,tabla,claves_duplicadas,claves_nulas
0,orders_clean,0,0
1,order_items_clean,0,0
2,customers_clean,0,0
3,products_clean,0,0
4,payments_clean,0,0


In [66]:
print("Integridad referencial:")

display(con.execute("""
SELECT
    'Pedidos sin cliente' AS validacion,
    COUNT(*) AS cantidad
FROM orders_clean o
LEFT JOIN customers_clean c
    ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL

UNION ALL

SELECT
    'Artículos sin pedido',
    COUNT(*)
FROM order_items_clean i
LEFT JOIN orders_clean o
    ON i.order_id = o.order_id
WHERE o.order_id IS NULL

UNION ALL

SELECT
    'Artículos sin producto',
    COUNT(*)
FROM order_items_clean i
LEFT JOIN products_clean p
    ON i.product_id = p.product_id
WHERE p.product_id IS NULL

UNION ALL

SELECT
    'Pagos sin pedido',
    COUNT(*)
FROM payments_clean p
LEFT JOIN orders_clean o
    ON p.order_id = o.order_id
WHERE o.order_id IS NULL
""").df())

Integridad referencial:


,validacion,cantidad
0,Pedidos sin cliente,0
1,Artículos sin pedido,0
2,Artículos sin producto,0
3,Pagos sin pedido,0


In [67]:
print("Valores inválidos restantes:")

display(con.execute("""
SELECT
    (SELECT COUNT(*)
     FROM order_items_clean
     WHERE price <= 0) AS precios_invalidos,

    (SELECT COUNT(*)
     FROM order_items_clean
     WHERE freight_value < 0) AS fletes_invalidos,

    (SELECT COUNT(*)
     FROM payments_clean
     WHERE payment_value < 0) AS pagos_negativos,

    (SELECT COUNT(*)
     FROM orders_clean
     WHERE order_status = 'canceled'
       AND order_delivered_customer_date IS NOT NULL
    ) AS cancelados_con_fecha_entrega
""").df())

Valores inválidos restantes:


,precios_invalidos,fletes_invalidos,pagos_negativos,cancelados_con_fecha_entrega
0,0,0,0,0


In [68]:
print("Estructura y tipos de datos:")

for tabla in [
    'orders_clean',
    'order_items_clean',
    'customers_clean',
    'payments_clean',
    'products_clean'
]:
    print(f"\n{tabla}:")
    display(con.execute(f"DESCRIBE {tabla}").df())

Estructura y tipos de datos:

orders_clean:


,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_purchase_timestamp,TIMESTAMP,YES,None,None,None
4,order_approved_at,TIMESTAMP,YES,None,None,None
5,order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
6,order_delivered_customer_date,TIMESTAMP,YES,None,None,None
7,order_estimated_delivery_date,TIMESTAMP,YES,None,None,None



order_items_clean:


,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,order_item_id,INTEGER,YES,None,None,None
2,product_id,VARCHAR,YES,None,None,None
3,seller_id,VARCHAR,YES,None,None,None
4,shipping_limit_date,TIMESTAMP,YES,None,None,None
5,price,DOUBLE,YES,None,None,None
6,freight_value,DOUBLE,YES,None,None,None



customers_clean:


,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,YES,None,None,None
1,customer_unique_id,VARCHAR,YES,None,None,None
2,customer_zip_code_prefix,VARCHAR,YES,None,None,None
3,customer_city,VARCHAR,YES,None,None,None
4,customer_state,VARCHAR,YES,None,None,None



payments_clean:


,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,payment_sequential,INTEGER,YES,None,None,None
2,payment_type,VARCHAR,YES,None,None,None
3,payment_installments,INTEGER,YES,None,None,None
4,payment_value,DOUBLE,YES,None,None,None



products_clean:


,column_name,column_type,null,key,default,extra
0,product_id,VARCHAR,YES,None,None,None
1,product_category_name,VARCHAR,YES,None,None,None
2,product_name_lenght,INTEGER,YES,None,None,None
3,product_description_lenght,INTEGER,YES,None,None,None
4,product_photos_qty,INTEGER,YES,None,None,None
5,product_weight_g,DOUBLE,YES,None,None,None
6,product_length_cm,DOUBLE,YES,None,None,None
7,product_height_cm,DOUBLE,YES,None,None,None
8,product_width_cm,DOUBLE,YES,None,None,None


In [69]:
print("Conteo final de tablas CLEAN:")

display(con.execute("""
SELECT 'orders_clean' AS tabla, COUNT(*) AS registros
FROM orders_clean

UNION ALL
SELECT 'order_items_clean', COUNT(*)
FROM order_items_clean

UNION ALL
SELECT 'customers_clean', COUNT(*)
FROM customers_clean

UNION ALL
SELECT 'payments_clean', COUNT(*)
FROM payments_clean

UNION ALL
SELECT 'products_clean', COUNT(*)
FROM products_clean
""").df())

Conteo final de tablas CLEAN:


,tabla,registros
0,orders_clean,99441
1,order_items_clean,112650
2,customers_clean,99441
3,payments_clean,103886
4,products_clean,32951


In [70]:
#Se realizó la validación final de las tablas CLEAN mediante consultas SQL en DuckDB. 
#Se verificó la unicidad de las claves, la ausencia de identificadores nulos, la integridad referencial entre las tablas y la existencia de valores inválidos en los campos numéricos y estados de pedido.

#Los resultados mostraron que no existen claves duplicadas ni nulas, tampoco registros huérfanos en las relaciones evaluadas. 
#Los precios y fletes inválidos fueron tratados durante las actividades anteriores y no se encontraron valores negativos restantes. 
#Los conteos finales de las cinco tablas coinciden con los registros esperados después de la limpieza.

#Se conservaron los valores nulos derivados del tratamiento de datos inválidos y de la ausencia de información, sin reemplazarlos por valores artificiales.

In [71]:
#quality report

con.execute("""
CREATE OR REPLACE TABLE quality_report (
    nombre_regla VARCHAR,
    tabla_evaluada VARCHAR,
    registros_evaluados BIGINT,
    registros_invalidos BIGINT,
    porcentaje_error DOUBLE,
    fecha_ejecucion TIMESTAMP
)
""")

fecha = datetime.now()

def insertar_regla(nombre, tabla, total, invalidos):
    pct = round((invalidos / total) * 100, 4) if total > 0 else 0
    con.execute("""
        INSERT INTO quality_report VALUES (?, ?, ?, ?, ?, ?)
    """, [nombre, tabla, total, invalidos, pct, fecha])

# 01 Duplicados orders
total = con.execute("SELECT COUNT(*) FROM orders_clean").fetchone()[0]
inv = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT order_id FROM orders_clean GROUP BY order_id HAVING COUNT(*) > 1
    )
""").fetchone()[0]
insertar_regla('01_duplicados', 'orders_clean', total, inv)

# 01 Duplicados order_items
total = con.execute("SELECT COUNT(*) FROM order_items_clean").fetchone()[0]
inv = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT order_id, order_item_id FROM order_items_clean
        GROUP BY 1,2 HAVING COUNT(*) > 1
    )
""").fetchone()[0]
insertar_regla('01_duplicados', 'order_items_clean', total, inv)

# 06 Precios inválidos (null)
total = con.execute("SELECT COUNT(*) FROM order_items_clean").fetchone()[0]
inv = con.execute("SELECT COUNT(*) FROM order_items_clean WHERE price IS NULL").fetchone()[0]
insertar_regla('06_precios_invalidos', 'order_items_clean', total, inv)

# 05 Fechas inconsistentes 
total = con.execute("SELECT COUNT(*) FROM orders_clean").fetchone()[0]
inv = con.execute("""
    SELECT COUNT(*) FROM orders_clean
    WHERE order_delivered_customer_date < order_purchase_timestamp
""").fetchone()[0]
insertar_regla('05_fechas_inconsistentes', 'orders_clean', total, inv)

# 11 Códigos postales nulos 
total = con.execute("SELECT COUNT(*) FROM customers_clean").fetchone()[0]
inv = con.execute("SELECT COUNT(*) FROM customers_clean WHERE customer_zip_code_prefix IS NULL").fetchone()[0]
insertar_regla('11_codigos_postales', 'customers_clean', total, inv)

# 13 Métodos desconocidos / otro
total = con.execute("SELECT COUNT(*) FROM payments_clean").fetchone()[0]
inv = con.execute("""
    SELECT COUNT(*) FROM payments_clean
    WHERE payment_type IN ('otro', 'desconocido')
""").fetchone()[0]
insertar_regla('13_metodos_pago', 'payments_clean', total, inv)

print("quality_report generada")
display(con.execute("SELECT * FROM quality_report ORDER BY nombre_regla").df())

quality_report generada


,nombre_regla,tabla_evaluada,registros_evaluados,registros_invalidos,porcentaje_error,fecha_ejecucion
0,01_duplicados,orders_clean,99441,0,0.0000,2026-11-13 15:07:58.708731
1,01_duplicados,order_items_clean,112650,0,0.0000,2026-11-13 15:07:58.708731
2,05_fechas_inconsistentes,orders_clean,99441,0,0.0000,2026-11-13 15:07:58.708731
3,06_precios_invalidos,order_items_clean,112650,60,0.0533,2026-11-13 15:07:58.708731
4,11_codigos_postales,customers_clean,99441,15,0.0151,2026-11-13 15:07:58.708731
5,13_metodos_pago,payments_clean,103886,20,0.0193,2026-11-13 15:07:58.708731


In [72]:
# hasta aqui mi informe tulio

In [73]:
# exportar a parquet

con.execute("COPY orders_clean TO 'orders_clean.parquet' (FORMAT PARQUET)")
con.execute("COPY order_items_clean TO 'order_items_clean.parquet' (FORMAT PARQUET)")
con.execute("COPY customers_clean TO 'customers_clean.parquet' (FORMAT PARQUET)")
con.execute("COPY payments_clean TO 'payments_clean.parquet' (FORMAT PARQUET)")
con.execute("COPY products_clean TO 'products_clean.parquet' (FORMAT PARQUET)")

print("Archivos Parquet generados")

Archivos Parquet generados


In [74]:
#con.close()